# Notebook 01: Core Types

**Purpose:** Develop and test fundamental types for ERPNext migration

**Contents:**
- Money class (currency, precision, validation)
- Account reference type
- Tax rate calculator
- Fiscal period validator

**Philosophy:** Build from primitives. Every higher-level type depends on these foundations.

## Setup

In [1]:
# Add src to path for development
import sys
from pathlib import Path

# Repository root
REPO_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
SRC_DIR = REPO_ROOT / 'src'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Repository root: {REPO_ROOT}")
print(f"Source directory: {SRC_DIR}")

Repository root: /home/jovyan/work/ERP/emt
Source directory: /home/jovyan/work/ERP/emt/src


In [2]:
# Standard library
from decimal import Decimal
from typing import List

# Third party
import pandas as pd

# Auto-reload modules during development
%load_ext autoreload
%autoreload 2

## Money Class

**Design goals:**
1. Immutable (thread-safe, prevents accidental modification)
2. Decimal precision (no float rounding errors)
3. Currency-aware (prevents mixing currencies)
4. ERPNext-compatible output format

**Why Decimal over float?**
```python
# Float precision issues
0.1 + 0.2 == 0.3  # False in Python!

# Decimal is exact
Decimal('0.1') + Decimal('0.2') == Decimal('0.3')  # True
```

In [3]:
from core.money import Money, get_currency_precision

### Test 1: Basic creation and validation

In [4]:
# Create money from different input types
m1 = Money(100, "KES")           # Integer
m2 = Money(100.50, "KES")        # Float
m3 = Money("100.50", "KES")      # String
m4 = Money(Decimal('100.50'), "KES")  # Decimal

print("Created money objects:")
print(f"From int:     {m1} = {m1.amount}")
print(f"From float:   {m2} = {m2.amount}")
print(f"From string:  {m3} = {m3.amount}")
print(f"From Decimal: {m4} = {m4.amount}")

assert m2 == m3 == m4, "Different input types should create equal Money"

Created money objects:
From int:     KES 100.00 = 100.00
From float:   KES 100.50 = 100.50
From string:  KES 100.50 = 100.50
From Decimal: KES 100.50 = 100.50


### Test 2: Rounding behavior

In [5]:
# Test rounding (ROUND_HALF_UP is standard accounting practice)
test_cases = [
    ("99.994", "99.99"),   # Round down
    ("99.995", "100.00"),  # Round up
    ("99.999", "100.00"),  # Round up
    ("0.005", "0.01"),     # Round up from exactly half
]

print("Rounding tests (2 decimal precision):")
for input_val, expected in test_cases:
    m = Money(input_val, "KES")
    result = str(m.amount)
    status = "✓" if result == expected else "✗"
    print(f"{status} {input_val:>10} → {result:>10} (expected {expected})")
    assert result == expected, f"Rounding failed for {input_val}"

Rounding tests (2 decimal precision):
✓     99.994 →      99.99 (expected 99.99)
✓     99.995 →     100.00 (expected 100.00)
✓     99.999 →     100.00 (expected 100.00)
✓      0.005 →       0.01 (expected 0.01)


### Test 3: Arithmetic operations

In [6]:
# Addition
m1 = Money(100, "KES")
m2 = Money(50.50, "KES")
m_sum = m1 + m2
print(f"Addition: {m1} + {m2} = {m_sum}")
assert m_sum.amount == Decimal('150.50')

# Subtraction
m_diff = m1 - m2
print(f"Subtraction: {m1} - {m2} = {m_diff}")
assert m_diff.amount == Decimal('49.50')

# Multiplication
m_mult = m1 * 1.5
print(f"Multiplication: {m1} × 1.5 = {m_mult}")
assert m_mult.amount == Decimal('150.00')

# Division
m_div = m1 / 4
print(f"Division: {m1} ÷ 4 = {m_div}")
assert m_div.amount == Decimal('25.00')

# Negation
m_neg = -m1
print(f"Negation: -{m1} = {m_neg}")
assert m_neg.amount == Decimal('-100.00')

print("\n✓ All arithmetic operations passed")

Addition: KES 100.00 + KES 50.50 = KES 150.50
Subtraction: KES 100.00 - KES 50.50 = KES 49.50
Multiplication: KES 100.00 × 1.5 = KES 150.00
Division: KES 100.00 ÷ 4 = KES 25.00
Negation: -KES 100.00 = KES -100.00

✓ All arithmetic operations passed


### Test 4: Currency mismatch protection

In [7]:
m_kes = Money(100, "KES")
m_usd = Money(100, "USD")

# Should raise ValueError
try:
    result = m_kes + m_usd
    print("✗ Should have raised ValueError for currency mismatch")
    assert False, "Currency mismatch not detected"
except ValueError as e:
    print(f"✓ Currency mismatch correctly detected: {e}")

# Comparison should also fail
try:
    result = m_kes > m_usd
    print("✗ Should have raised ValueError for currency comparison")
    assert False, "Currency comparison mismatch not detected"
except ValueError as e:
    print(f"✓ Currency comparison mismatch correctly detected: {e}")

✓ Currency mismatch correctly detected: Cannot add KES and USD
✓ Currency comparison mismatch correctly detected: Cannot compare KES with USD


### Test 5: ERPNext format compatibility

In [8]:
# ERPNext API expects float for currency fields
m = Money("123.456", "KES")  # Will round to 123.46

erpnext_value = m.to_erpnext_format()
print(f"Money object: {m}")
print(f"ERPNext format: {erpnext_value}")
print(f"Type: {type(erpnext_value)}")

assert isinstance(erpnext_value, float), "ERPNext format should be float"
assert erpnext_value == 123.46, "Value should be rounded correctly"

print("\n✓ ERPNext format conversion passed")

Money object: KES 123.46
ERPNext format: 123.46
Type: <class 'float'>

✓ ERPNext format conversion passed


### Test 6: Edge cases

In [9]:
# Zero
m_zero = Money.zero("KES")
print(f"Zero: {m_zero}, is_zero={m_zero.is_zero()}")
assert m_zero.is_zero()

# Negative
m_negative = Money(-50, "KES")
print(f"Negative: {m_negative}, is_negative={m_negative.is_negative()}")
assert m_negative.is_negative()
assert not m_negative.is_positive()

# From ERPNext None value
m_from_none = Money.from_erpnext(None, "KES")
print(f"From None: {m_from_none}")
assert m_from_none.is_zero()

# Division by zero should raise
try:
    m = Money(100, "KES") / 0
    print("✗ Should have raised ZeroDivisionError")
    assert False
except ZeroDivisionError:
    print("✓ Division by zero correctly raises error")

print("\n✓ All edge cases handled correctly")

Zero: KES 0.00, is_zero=True
Negative: KES -50.00, is_negative=True
From None: KES 0.00
✓ Division by zero correctly raises error

✓ All edge cases handled correctly


### Test 7: Real-world scenario - Invoice totals

In [10]:
# Simulate calculating invoice total from line items
line_items = [
    {"description": "Event Venue Hire", "amount": Money(15000, "KES")},
    {"description": "Room Accommodation (2 nights)", "amount": Money(14000, "KES")},
    {"description": "Wellness Session", "amount": Money(5000, "KES")},
]

# Calculate subtotal
subtotal = sum((item["amount"] for item in line_items), Money.zero("KES"))
print(f"Subtotal: {subtotal}")

# Apply tax (16% VAT in Kenya)
tax_rate = Decimal('0.16')
tax_amount = subtotal * tax_rate
print(f"Tax (16%): {tax_amount}")

# Calculate total
total = subtotal + tax_amount
print(f"Total: {total}")

# Verify calculations
assert subtotal.amount == Decimal('34000.00')
assert tax_amount.amount == Decimal('5440.00')
assert total.amount == Decimal('39440.00')

print("\n✓ Invoice calculation test passed")

# Show ERPNext API payload format
api_payload = {
    "doctype": "Sales Invoice",
    "total": subtotal.to_erpnext_format(),
    "total_taxes_and_charges": tax_amount.to_erpnext_format(),
    "grand_total": total.to_erpnext_format(),
}

print("\nERPNext API payload:")
for key, value in api_payload.items():
    if key != "doctype":
        print(f"  {key}: {value}")

Subtotal: KES 34,000.00
Tax (16%): KES 5,440.00
Total: KES 39,440.00

✓ Invoice calculation test passed

ERPNext API payload:
  total: 34000.0
  total_taxes_and_charges: 5440.0
  grand_total: 39440.0


## Summary

**Money class validation complete:**
- ✓ Creates from int, float, string, Decimal
- ✓ Rounds correctly (ROUND_HALF_UP)
- ✓ Arithmetic operations work
- ✓ Currency mismatch protection
- ✓ ERPNext format conversion
- ✓ Edge cases handled
- ✓ Real-world invoice scenario

**Next steps:**
1. Extract Money class to `src/core/money.py` ✓ (already done)
2. Create unit tests in `tests/test_core/test_money.py`
3. Develop Account type
4. Develop Tax type
5. Develop FiscalPeriod type

## Next: Account Type

**Requirements:**
- Account code validation (format checking)
- Account type validation (Asset, Liability, Income, Expense, Equity)
- Account hierarchy support (parent-child relationships)
- ERPNext Chart of Accounts compatibility

**To be developed in next cell...**